# Phân Tích Insights Bằng Claude AI
## Tổng hợp nhận xét actionable cho CEO từ kết quả 3 notebooks trước

**Mô hình:** claude-haiku-4-5-20251001

**Cách dùng:** Mỗi cell gọi `llm_insight()` với tóm tắt kết quả từ Q1/Q2/Q3 và nhận 4 bullet points tiếng Việt.

Nếu không có ANTHROPIC_API_KEY, mỗi cell sẽ in fallback text.

In [1]:
# Cell 1: Setup client & helper function
import os
import pandas as pd
import numpy as np

try:
    import anthropic
    client = anthropic.Anthropic()  # đọc từ ANTHROPIC_API_KEY env var
    LLM_AVAILABLE = True
    print('Anthropic client khởi tạo thành công.')
except Exception as e:
    LLM_AVAILABLE = False
    print(f'[INFO] Anthropic không khả dụng: {e}')
    print('Sẽ dùng fallback text.')

def llm_insight(section: str, summary_text: str) -> str:
    """Gọi Claude để tạo insight tiếng Việt cho CEO."""
    if not LLM_AVAILABLE:
        return f"[LLM không khả dụng — vui lòng set ANTHROPIC_API_KEY]\n\nFallback summary cho {section}:\n{summary_text[:300]}"
    try:
        msg = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=400,
            messages=[{"role": "user", "content":
                f"Bạn là chuyên gia phân tích B2B cho công ty phân phối xe đạp.\n\n"
                f"Kết quả {section}:\n{summary_text}\n\n"
                f"Viết 4 bullet points nhận xét actionable bằng tiếng Việt cho CEO."
            }]
        )
        return msg.content[0].text
    except Exception as e:
        return f"[LLM không khả dụng: {e}]"

Anthropic client khởi tạo thành công.


In [2]:
# Cell 2: Load dữ liệu để tổng hợp thực tế
df = pd.read_csv('../data/fact_sales.csv', low_memory=False)
df['order_date'] = pd.to_datetime(df['order_date'])
df['product_code'] = df['product_code'].astype(str).str.strip().str.zfill(15)
df['seg2_group'] = df['product_code'].str[6:9]
df['year_month'] = df['order_date'].dt.to_period('M')
df = df[(df['year_month'].astype(str) != '2026-03') & (df['seg2_group'] != '00U')]

SEG2_MAP = {'002': 'Xe thường', '003': 'Địa hình', '004': 'Gấp', '005': 'Điện'}
df['group_name'] = df['seg2_group'].map(SEG2_MAP).fillna('Khác')
print(f'Dữ liệu: {len(df):,} dòng')

Dữ liệu: 34,062 dòng


In [3]:
# Cell 3: Tóm tắt EDA cho LLM
monthly_rev = df.groupby('year_month')['line_total'].sum()
group_rev = df.groupby('group_name')['line_total'].sum().sort_values(ascending=False)
top_dealers = df.groupby('customer_name')['line_total'].sum().sort_values(ascending=False).head(5)

eda_summary = f"""
Tổng doanh thu 5 tháng sạch: {df['line_total'].sum()/1e9:.1f} tỷ VND
Doanh thu theo tháng (tỷ VND): { {str(k): round(v/1e9,1) for k,v in monthly_rev.items()} }
Doanh thu theo nhóm SP: { {k: round(v/1e9,1) for k,v in group_rev.items()} }
Top 5 đại lý: { {str(k)[:30]: round(v/1e9,1) for k,v in top_dealers.items()} }
Số đại lý hoạt động: {df['customer_code'].nunique()}
Số SKU: {df['product_code'].nunique()}
Tháng đỉnh: {monthly_rev.idxmax()} ({monthly_rev.max()/1e9:.1f}B)
"""

print('=== INSIGHT 1: EDA OVERVIEW ===')
insight_eda = llm_insight('phân tích EDA tổng quan', eda_summary)
print(insight_eda)

=== INSIGHT 1: EDA OVERVIEW ===
[LLM không khả dụng: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"]


In [4]:
# Cell 4: Tóm tắt kết quả dự báo Q1 cho LLM
# Tái chạy dự báo nhanh để lấy kết quả
from sklearn.linear_model import LinearRegression as _LR
import lightgbm as _lgb
import warnings; warnings.filterwarnings('ignore')

monthly = df.groupby('year_month', as_index=False)['line_total'].sum().sort_values('year_month')
monthly['month_num'] = np.arange(len(monthly))
monthly['month'] = monthly['year_month'].astype(str).str[-2:].astype(int)
monthly['month_sin'] = np.sin(2 * np.pi * monthly['month'] / 12)
monthly['month_cos'] = np.cos(2 * np.pi * monthly['month'] / 12)
monthly['lag_1'] = monthly['line_total'].shift(1)
monthly['lag_2'] = monthly['line_total'].shift(2)
monthly['rolling_mean_2'] = monthly['line_total'].shift(1).rolling(2).mean()
monthly_feat = monthly.dropna().copy()
FEATURES = ['month_num','month_sin','month_cos','lag_1','lag_2','rolling_mean_2']

lgbm = _lgb.LGBMRegressor(n_estimators=100, max_depth=3, random_state=42, verbose=-1)
lgbm.fit(monthly_feat[FEATURES].values, monthly_feat['line_total'].values)

all_vals = list(monthly['line_total'].values)
q2_forecasts = []
for i, mn in enumerate([4, 5, 6]):
    nm = len(all_vals)
    feat = np.array([[nm, np.sin(2*np.pi*mn/12), np.cos(2*np.pi*mn/12),
                      all_vals[-1], all_vals[-2], np.mean(all_vals[-2:])]])
    p = lgbm.predict(feat)[0]
    q2_forecasts.append(p)
    all_vals.append(p)

q2_summary = f"""
Mô hình tốt nhất: LightGBM (so sánh với Naive/LR/ETS)
Dự báo Q2/2026:
  Tháng 4/2026: {q2_forecasts[0]/1e9:.2f} tỷ VND
  Tháng 5/2026: {q2_forecasts[1]/1e9:.2f} tỷ VND
  Tháng 6/2026: {q2_forecasts[2]/1e9:.2f} tỷ VND
  Tổng Q2/2026: {sum(q2_forecasts)/1e9:.2f} tỷ VND
Q1/2026 thực tế: {df[df['year_month'].astype(str).isin(['2026-01','2026-02'])]['line_total'].sum()/1e9:.2f} tỷ VND (2 tháng)
Xu hướng: doanh thu ổn định, lag-1 là feature quan trọng nhất
"""

print('=== INSIGHT 2: DỰ BÁO DOANH THU Q2/2026 ===')
insight_q1 = llm_insight('dự báo doanh thu Q2/2026', q2_summary)
print(insight_q1)

=== INSIGHT 2: DỰ BÁO DOANH THU Q2/2026 ===
[LLM không khả dụng: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"]


In [5]:
# Cell 5: Tóm tắt kết quả màu sắc & SKU cho LLM
df['seg3_color'] = df['product_code'].str[9:12]
color_monthly = df.groupby(['year_month','seg3_color'])['quantity'].sum().unstack(fill_value=0)
color_monthly.index = [str(i) for i in color_monthly.index]
color_share = color_monthly.div(color_monthly.sum(axis=1), axis=0) * 100
top_colors = color_monthly.sum().sort_values(ascending=False).head(5).index.tolist()

share_2025 = color_share.loc[color_share.index.isin(['2025-01','2025-02','2025-03'])][top_colors].mean()
share_2026 = color_share.loc[color_share.index.isin(['2026-01','2026-02'])][top_colors].mean()
yoy_change = (share_2026 - share_2025).round(1)

# SKU slopes
sku_monthly = df.groupby(['year_month','product_code'])['quantity'].sum().unstack(fill_value=0)
sku_monthly.index = [str(i) for i in sku_monthly.index]
X_s = np.arange(len(sku_monthly)).reshape(-1,1)
slopes = {sku: _LR().fit(X_s, sku_monthly[sku].values).coef_[0] for sku in sku_monthly.columns if sku_monthly[sku].sum() > 0}
n_declining = sum(1 for v in slopes.values() if v < 0)
n_growing = sum(1 for v in slopes.values() if v > 0)

color_summary = f"""
Top 5 màu sắc và thay đổi thị phần YoY (percentage points):
{yoy_change.to_dict()}
Màu tăng mạnh nhất: {yoy_change.idxmax()} (+{yoy_change.max():.1f}pp)
Màu giảm mạnh nhất: {yoy_change.idxmin()} ({yoy_change.min():.1f}pp)
Tổng SKU phân tích: {len(slopes)}
SKU có xu hướng tăng: {n_growing} ({n_growing/len(slopes)*100:.0f}%)
SKU có xu hướng giảm: {n_declining} ({n_declining/len(slopes)*100:.0f}%)
"""

print('=== INSIGHT 3: XU HƯỚNG MÀU SẮC & SKU ===')
insight_q2 = llm_insight('xu hướng màu sắc và SKU', color_summary)
print(insight_q2)

=== INSIGHT 3: XU HƯỚNG MÀU SẮC & SKU ===
[LLM không khả dụng: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"]


In [6]:
# Cell 6: Tóm tắt kết quả đại lý & churn cho LLM
from sklearn.linear_model import LogisticRegression as _LogReg
from sklearn.preprocessing import StandardScaler as _SS

REFERENCE_DATE = pd.Timestamp('2026-03-01')
rfm = df.groupby('customer_code').agg(
    recency=('order_date', lambda x: (REFERENCE_DATE - x.max()).days),
    frequency=('so_number', 'nunique'),
    monetary=('line_total', 'sum')
).reset_index()

# Logistic regression
df_jan = df[df['year_month'].astype(str) <= '2026-01']
rfm_jan = df_jan.groupby('customer_code').agg(
    recency_jan=('order_date', lambda x: (pd.Timestamp('2026-02-01') - x.max()).days),
    frequency_jan=('so_number', 'nunique'),
    monetary_jan=('line_total', 'sum')
).reset_index()
feb_buyers = set(df[df['year_month'].astype(str) == '2026-02']['customer_code'].unique())
rfm_jan['reorder_feb'] = rfm_jan['customer_code'].isin(feb_buyers).astype(int)

X_lr = rfm_jan[['recency_jan','frequency_jan','monetary_jan']].fillna(0).values
y_lr = rfm_jan['reorder_feb'].values
scaler = _SS(); X_lr_s = scaler.fit_transform(X_lr)
log_reg = _LogReg(class_weight='balanced', random_state=42, max_iter=500)
log_reg.fit(X_lr_s, y_lr)
acc = (log_reg.predict(X_lr_s) == y_lr).mean()

df_feb = df[df['year_month'].astype(str) <= '2026-02']
rfm_feb = df_feb.groupby('customer_code').agg(
    recency_feb=('order_date', lambda x: (pd.Timestamp('2026-03-01') - x.max()).days),
    frequency_feb=('so_number', 'nunique'),
    monetary_feb=('line_total', 'sum')
).reset_index()
X_pred = rfm_feb[['recency_feb','frequency_feb','monetary_feb']].fillna(0).values
X_pred_s = scaler.transform(X_pred)
probs = log_reg.predict_proba(X_pred_s)[:,1]
n_high_risk = (probs < 0.3).sum()
n_low_risk  = (probs >= 0.6).sum()

dealer_summary = f"""
Tổng số đại lý phân tích: {len(rfm)}
Logistic Regression (Jan→Feb transition):
  Accuracy: {acc:.2f}
  Đại lý HIGH RISK (prob < 0.3): {n_high_risk} ({n_high_risk/len(rfm_feb)*100:.0f}%)
  Đại lý LOW RISK (prob ≥ 0.6): {n_low_risk} ({n_low_risk/len(rfm_feb)*100:.0f}%)
RFM thống kê:
  Recency trung bình: {rfm['recency'].mean():.0f} ngày
  Frequency trung bình: {rfm['frequency'].mean():.1f} đơn
  Monetary trung bình: {rfm['monetary'].mean()/1e6:.0f} triệu VND
"""

print('=== INSIGHT 4: PHÂN TÍCH ĐẠI LÝ & CHURN ===')
insight_q3 = llm_insight('phân tích đại lý và dự báo churn', dealer_summary)
print(insight_q3)

=== INSIGHT 4: PHÂN TÍCH ĐẠI LÝ & CHURN ===
[LLM không khả dụng: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"]


In [7]:
# Cell 7: Insight tổng hợp chiến lược Q2/2026
strategic_summary = f"""
Tổng hợp kết quả phân tích cho CEO:

1. DỰ BÁO DOANH THU Q2/2026:
   - Tháng 4: {q2_forecasts[0]/1e9:.1f}B | Tháng 5: {q2_forecasts[1]/1e9:.1f}B | Tháng 6: {q2_forecasts[2]/1e9:.1f}B
   - Tổng Q2: {sum(q2_forecasts)/1e9:.1f} tỷ VND
   - Mô hình LightGBM, lag-1 feature quan trọng nhất

2. SẢN PHẨM:
   - Xe thường (002) chiếm >70% doanh thu
   - {n_growing} SKU tăng trưởng, {n_declining} SKU suy giảm
   - Màu tăng mạnh nhất: {yoy_change.idxmax()}

3. ĐẠI LÝ:
   - {n_high_risk} đại lý HIGH RISK cần chăm sóc khẩn cấp
   - {n_low_risk} đại lý ổn định, cần maintain relationship
   - Recency trung bình: {rfm['recency'].mean():.0f} ngày
"""

print('=== INSIGHT 5: CHIẾN LƯỢC TỔNG QUAN Q2/2026 ===')
insight_strategy = llm_insight('chiến lược tổng quan Q2/2026', strategic_summary)
print(insight_strategy)

=== INSIGHT 5: CHIẾN LƯỢC TỔNG QUAN Q2/2026 ===
[LLM không khả dụng: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"]


In [8]:
# Cell 8: In tổng hợp tất cả insights
print('=' * 70)
print('TỔNG HỢP TẤT CẢ INSIGHTS CHO CEO')
print('=' * 70)
print()
print('## 1. TỔNG QUAN KINH DOANH')
print(insight_eda)
print()
print('## 2. DỰ BÁO DOANH THU Q2/2026')
print(insight_q1)
print()
print('## 3. XU HƯỚNG SẢN PHẨM')
print(insight_q2)
print()
print('## 4. QUẢN LÝ ĐẠI LÝ')
print(insight_q3)
print()
print('## 5. CHIẾN LƯỢC Q2/2026')
print(insight_strategy)

TỔNG HỢP TẤT CẢ INSIGHTS CHO CEO

## 1. TỔNG QUAN KINH DOANH
[LLM không khả dụng: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"]

## 2. DỰ BÁO DOANH THU Q2/2026
[LLM không khả dụng: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"]

## 3. XU HƯỚNG SẢN PHẨM
[LLM không khả dụng: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"]

## 4. QUẢN LÝ ĐẠI LÝ
[LLM không khả dụng: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"]

## 5. CHIẾN LƯỢC Q2

## Hướng Dẫn Sử Dụng

**Để kích hoạt Claude API:**
```bash
export ANTHROPIC_API_KEY='your-api-key'
jupyter nbconvert --to notebook --execute 04_llm_insights.ipynb
```

**Mô hình sử dụng:** `claude-haiku-4-5-20251001` (chi phí thấp, phù hợp tóm tắt ngắn)

**Tùy chỉnh:** Thay đổi `summary_text` trong mỗi cell để cập nhật kết quả mới từ các notebook trước.